In [13]:
import pandas as pd
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from scipy.stats import mannwhitneyu, norm
from scipy.stats import chi2_contingency
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import kendalltau
from scipy.stats import anderson
import plotly.graph_objects as go
rng = np.random.default_rng()
x = stats.uniform.rvs(size=75, random_state=rng)
sns.set(style="darkgrid")

#
df_todo = pd.read_csv('Banking_Transactions_USA_2023_2024.csv', nrows=5)  # solo 5 filas para ver
print(f'Columnas totales: {len(df_todo.columns)}')
print()
print('Lista de columnas:')
for i, col in enumerate(df_todo.columns, 1):
    print(f'  {i:>2}. {col}')

Columnas totales: 20

Lista de columnas:
   1. Transaction_ID
   2. Account_Number
   3. Transaction_Date
   4. Transaction_Amount
   5. Merchant_Name
   6. Transaction_Type
   7. Category
   8. City
   9. Country
  10. Payment_Method
  11. Customer_Age
  12. Customer_Gender
  13. Customer_Occupation
  14. Customer_Income
  15. Account_Balance
  16. Transaction_Status
  17. Fraud_Flag
  18. Discount_Applied
  19. Loyalty_Points_Earned
  20. Transaction_Description


In [14]:
#
df_bancos = pd.read_csv('Banking_Transactions_USA_2023_2024.csv')
mb_full = df_bancos.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset completo: {df_bancos.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in df_bancos.columns:
    dtype = str(df_bancos[col].dtype)
    nan_pct = df_bancos[col].isna().mean() * 100
    n_uniq  = df_bancos[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

Dataset completo: (5389, 20) · 5.3 MB

Columna                        Dtype           NaN%   Unicos
-----------------------------------------------------------------
Transaction_ID                 object          0.0%    5,389
Account_Number                 object          0.0%    5,389
Transaction_Date               object          0.0%    5,389
Transaction_Amount             float64         0.0%    5,369
Merchant_Name                  object          0.0%    4,880
Transaction_Type               object          0.0%        2
Category                       object          0.0%       14
City                           object          0.0%       10
Country                        object          0.0%        1
Payment_Method                 object          0.0%        5
Customer_Age                   int64           0.0%       53
Customer_Gender                object          0.0%        3
Customer_Occupation            object          0.0%      639
Customer_Income                float64   

In [15]:
#Cabecera: Primeros renglones de la base
df_bancos.head()

,Transaction_ID,Account_Number,Transaction_Date,Transaction_Amount,Merchant_Name,Transaction_Type,Category,City,Country,Payment_Method,Customer_Age,Customer_Gender,Customer_Occupation,Customer_Income,Account_Balance,Transaction_Status,Fraud_Flag,Discount_Applied,Loyalty_Points_Earned,Transaction_Description
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,IUPM04409079772781,2023-11-05 15:54:38,3198.94,Houston Group,Debit,Transport,Phoenix,USA,Online Transfer,55,Others,Quality manager,80466.03,350.28,Failed,No,True,304,Recently company detail form range a.
1,23b8c1e9-3924-46de-beb1-3b9046685257,BLAT22216107051843,2024-04-21 22:21:55,129.93,Anderson-Phillips,Credit,Grocery,Philadelphia,USA,Debit Card,26,Others,Civil Service fast streamer,145574.25,9797.81,Pending,Yes,False,383,Anything son baby power heart will not up.
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,UTXA55295806601382,2023-07-17 13:25:56,1378.77,Jensen Group,Credit,Shopping,New York,USA,Debit Card,29,Others,"Pilot, airline",33447.18,12399.85,Failed,Yes,False,497,Form world around green bar environment pattern.
3,972a8469-1641-4f82-8b9d-2434e465e150,XICF70493862044851,2023-06-27 16:09:52,1119.94,"Nelson, Gomez and Rodriguez",Credit,Healthcare,Dallas,USA,Online Transfer,60,Male,"Radiographer, therapeutic",108801.45,16057.64,Failed,Yes,True,495,Order evening source these opportunity trade i...
4,17fc695a-07a0-4a6e-8822-e8f36c031199,KOSW19711121259020,2024-03-26 23:45:31,3683.67,Caldwell Group,Credit,Entertainment,San Jose,USA,E-Wallet,29,Others,Diplomatic Services operational officer,100985.12,14940.54,Failed,Yes,True,292,Exactly politics door suggest.


In [16]:
# Ultimos elementos de la base
df_bancos.tail()

,Transaction_ID,Account_Number,Transaction_Date,Transaction_Amount,Merchant_Name,Transaction_Type,Category,City,Country,Payment_Method,Customer_Age,Customer_Gender,Customer_Occupation,Customer_Income,Account_Balance,Transaction_Status,Fraud_Flag,Discount_Applied,Loyalty_Points_Earned,Transaction_Description
5384,31e43850-86a8-4b2b-9c69-ee49712c3126,LXVL92532425475957,2023-05-01 07:20:11,842.16,Navarro Group,Credit,Savings,Phoenix,USA,Cash,20,Male,"Loss adjuster, chartered",75036.50,8747.79,Pending,Yes,True,332,Sit could account happen organization hand act...
5385,5415668e-4f75-43d1-9610-7189016ff44c,MFJS29896187528828,2023-09-15 03:49:42,1996.73,Banks-Reed,Credit,Travel,Phoenix,USA,Credit Card,28,Male,Accommodation manager,107479.71,2698.54,Failed,Yes,True,60,Soldier per crime nothing improve structure ow...
5386,ea725584-5e6e-487a-a939-f79018177e4a,DOBF62458522431672,2024-11-30 00:58:14,1510.33,Brown PLC,Debit,Utilities,Dallas,USA,Credit Card,23,Male,"Administrator, education",33759.96,476.93,Success,No,True,475,Magazine reality share office doctor managemen...
5387,03539b9a-ebeb-4d98-80b7-10a640f66fde,BRHX88846108622659,2023-08-09 00:31:15,3732.02,Townsend-Bartlett,Debit,Savings,Chicago,USA,Cash,40,Female,Music therapist,32706.74,7167.05,Success,Yes,True,482,Probably ago western approach.
5388,a5b01639-0895-4db1-85e5-26e19b559178,CTJA66942505562975,2024-03-16 09:44:09,3763.38,"Molina, Smith and King",Debit,Utilities,Houston,USA,Debit Card,38,Male,Structural engineer,106270.94,2197.74,Success,Yes,False,152,Social suddenly lead.


In [17]:
# Resumen de las variables numericas

resumen_numericas = df_bancos.describe()
resumen_numericas

,Transaction_Amount,Customer_Age,Customer_Income,Account_Balance,Loyalty_Points_Earned
count,5389.000000,5389.000000,5389.000000,5389.000000,5389.000000
mean,2504.649200,44.023567,85802.754023,10096.510087,249.897384
std,1426.745115,15.239148,37343.604534,5732.846891,145.378373
min,5.460000,18.000000,20028.940000,101.720000,0.000000
25%,1283.340000,31.000000,53809.800000,5159.450000,122.000000
50%,2521.670000,44.000000,85636.380000,10077.780000,251.000000
75%,3715.920000,57.000000,118092.640000,15076.010000,377.000000
max,4999.540000,70.000000,149970.500000,19993.040000,500.000000


In [18]:
# Resumen de las variables discretas

resumen_discretas = df_bancos.describe(include='object')
resumen_discretas

,Transaction_ID,Account_Number,Transaction_Date,Merchant_Name,Transaction_Type,Category,City,Country,Payment_Method,Customer_Gender,Customer_Occupation,Transaction_Status,Fraud_Flag,Transaction_Description
count,5389,5389,5389,5389,5389,5389,5389,5389,5389,5389,5389,5389,5389,5389
unique,5389,5389,5389,4880,2,14,10,1,5,3,639,3,2,5389
top,bdd640fb-0667-4ad1-9c80-317fa3b1799d,IUPM04409079772781,2023-11-05 15:54:38,Smith LLC,Debit,Utilities,Chicago,USA,E-Wallet,Female,Child psychotherapist,Pending,No,Recently company detail form range a.
freq,1,1,1,10,2708,417,589,5389,1100,1824,18,1811,2732,1


In [36]:
#
percentiles = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

# columnas numericas
columnas_analisis = ['Transaction_Amount', 'Customer_Age', 'Customer_Income', 'Account_Balance', 'Loyalty_Points_Earned']

#
df_percentiles = df_bancos[columnas_analisis].quantile(percentiles)

#
df_percentiles.index = [
    'Percentil 10 (P10)', 
    'Percentil 20 (P20)', 
    'Percentil 30 (P30)', 
    'Percentil 40 (P40)', 
    'Percentil 50 (P50)',
    'Percentil 60 (P60)', 
    'Percentil 70 (P70)', 
    'Percentil 80 (P80)', 
    'Percentil 90 (P90)'
]
# 
print(" REPORTE PERCENTILES - VARIABLES NUMERICAS")
df_percentiles.style.format("${:,.2f}")


 REPORTE PERCENTILES - VARIABLES NUMERICAS


,Transaction_Amount,Customer_Age,Customer_Income,Account_Balance,Loyalty_Points_Earned
Percentil 10 (P10),$519.45,$23.00,"$33,445.10","$2,043.31",$49.00
Percentil 20 (P20),"$1,017.71",$28.00,"$47,541.39","$4,115.05",$98.00
Percentil 30 (P30),"$1,519.80",$33.00,"$60,243.90","$6,257.16",$148.00
Percentil 40 (P40),"$2,019.48",$39.00,"$73,394.66","$8,326.80",$198.00
Percentil 50 (P50),"$2,521.67",$44.00,"$85,636.38","$10,077.78",$251.00
Percentil 60 (P60),"$3,013.97",$49.00,"$99,216.63","$12,136.19",$300.00
Percentil 70 (P70),"$3,480.21",$55.00,"$111,758.55","$14,064.19",$352.60
Percentil 80 (P80),"$3,967.22",$60.00,"$124,205.44","$16,030.41",$403.00
Percentil 90 (P90),"$4,476.50",$65.00,"$137,448.35","$17,928.00",$451.00


In [37]:
#
cuartiles = [0.25, 0.50, 0.75]

# columnas numericas
columnas_analisis = ['Transaction_Amount', 'Customer_Age', 'Customer_Income', 'Account_Balance', 'Loyalty_Points_Earned']

#
df_cuartiles = df_bancos[columnas_analisis].quantile(cuartiles)

#
df_cuartiles.index = [
    'Cuartil 1 (Q1)', 
    'Cuartil 2 (Q2)',
    'Cuartil 3 (Q3)'
]
# 
print(" REPORTE CUARTILES - VARIABLES NUMERICAS")
df_cuartiles.style.format("${:,.2f}")

 REPORTE CUARTILES - VARIABLES NUMERICAS


,Transaction_Amount,Customer_Age,Customer_Income,Account_Balance,Loyalty_Points_Earned
Cuartil 1 (Q1),"$1,283.34",$31.00,"$53,809.80","$5,159.45",$122.00
Cuartil 2 (Q2),"$2,521.67",$44.00,"$85,636.38","$10,077.78",$251.00
Cuartil 3 (Q3),"$3,715.92",$57.00,"$118,092.64","$15,076.01",$377.00
